# 04 - Text Classification com Fine-Tuning do BERTugues

Neste notebook, vamos realizar o **Fine-tuning** (ajuste fino) do modelo `ricardoz/BERTugues-base-portuguese-cased` utilizando a biblioteca `transformers` do Hugging Face.

Agora, permitimos configurar o **Pooling Strategy** (CLS vs Mean) e o **Tamanho da Amostra**, assim como fizemos no notebook anterior.

## 1. Instalando as bibliotecas necessárias

In [1]:
# !pip install transformers datasets evaluate accelerate scikit-learn torch pandas

import torch
import pandas as pd
import os
import urllib.request
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, BertModel, BertPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
import evaluate
import numpy as np

## 2. Configuração e Carregamento de Dados

In [2]:
# ================= CONFIGURAÇÃO =================
tamanho_amostra = 3000   # Quantas frases usar (Total ~130k)
pooling_strategy = 'mean' # 'cls' ou 'mean'
# ================================================

model_name = "ricardoz/BERTugues-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Verificando Dataset B2W...")
url_csv = "https://raw.githubusercontent.com/b2wdigital/b2w-reviews01/master/B2W-Reviews01.csv"
pasta_dataset = "datasets"
arquivo_csv_local = f"{pasta_dataset}/B2W-Reviews01.csv"

os.makedirs(pasta_dataset, exist_ok=True)

if not os.path.exists(arquivo_csv_local):
    print("Baixando CSV do GitHub...")
    urllib.request.urlretrieve(url_csv, arquivo_csv_local)
else:
    print("Cache local detectado.")

df_raw = pd.read_csv(arquivo_csv_local, sep=',', low_memory=False, on_bad_lines='skip')
df_limpo = df_raw.dropna(subset=['review_text', 'overall_rating'])

if tamanho_amostra > len(df_limpo):
    tamanho_amostra = len(df_limpo)
    
df = df_limpo.sample(n=tamanho_amostra, random_state=42).copy()
df['label'] = pd.to_numeric(df['overall_rating'], errors='coerce').apply(lambda x: 1 if x > 3 else 0)
df['text'] = df['review_text'].astype(str)

hf_dataset = Dataset.from_pandas(df[['text', 'label']])
hf_dataset = hf_dataset.remove_columns([col for col in hf_dataset.column_names if col not in ['text', 'label']])
train_test_split = hf_dataset.train_test_split(test_size=0.15, seed=42)

tokenized_datasets = train_test_split.map(
    lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=128), 
    batched=True
)

print(f"Dataset pronto! Pooling: {pooling_strategy} | Amostra: {tamanho_amostra}")

Verificando Dataset B2W...
Cache local detectado.


Map:   0%|          | 0/2550 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Dataset pronto! Pooling: mean | Amostra: 3000


## 3. Customização do Pooling no Fine-Tuning

Para suportar **Mean Pooling** durante o ajuste fino, criamos uma classe customizada que herda da arquitetura original, mas permite escolher como consolidar os tokens na camada de saída.

In [3]:
class BERTuguesForClassification(BertPreTrainedModel):
    def __init__(self, config, pooling_strategy='cls'):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.bert = BertModel(config)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.pooling_strategy = pooling_strategy
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, **kwargs)
        last_hidden_state = outputs.last_hidden_state

        if self.pooling_strategy == 'cls':
            pooled_output = last_hidden_state[:, 0, :]
        elif self.pooling_strategy == 'mean':
            mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
            sum_embeddings = torch.sum(last_hidden_state * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            pooled_output = sum_embeddings / sum_mask
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

# Instancia o modelo com a estratégia escolhida
model = BERTuguesForClassification.from_pretrained(model_name, num_labels=2, pooling_strategy=pooling_strategy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERTuguesForClassification LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

## 4. Fine-Tuning

In [4]:
training_args = TrainingArguments(
    output_dir=f"./bertugues-finetuned-{pooling_strategy}", 
    learning_rate=2e-5,               
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,    
    num_train_epochs=2,               # 2 épocas para teste rápido
    weight_decay=0.01,
    eval_strategy="epoch",      
    save_strategy="epoch",            
    logging_steps=50,                 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.314491,0.322514,0.875556
2,0.235032,0.318517,0.880000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=320, training_loss=0.305116169154644, metrics={'train_runtime': 35.6314, 'train_samples_per_second': 143.132, 'train_steps_per_second': 8.981, 'total_flos': 335466595584000.0, 'train_loss': 0.305116169154644, 'epoch': 2.0})

## 5. Inferência Otimizada em Lote (Batch)

In [5]:
def predict_batch(texts, batch_size=16):
    """
    Realiza predições em lotes para acelerar a inferência.
    """
    all_predictions = []
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = model(**inputs).logits
            
        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_predictions.extend(["Positivo" if p == 1 else "Negativo" for p in batch_preds])
        
    return all_predictions

testes = [
    "O produto é excelente, chegou muito rápido!",
    "Não gostei, o material é frágil e quebrou no primeiro dia.",
    "Serviço de entrega péssimo, mas o produto em si é razoável.",
    "Vale cada centavo, compraria novamente sem dúvida."
]

resultados = predict_batch(testes, batch_size=4)

for t, r in zip(testes, resultados):
    print(f"Texto: {t} | Predição ({pooling_strategy}): {r}")

Texto: O produto é excelente, chegou muito rápido! | Predição (mean): Positivo
Texto: Não gostei, o material é frágil e quebrou no primeiro dia. | Predição (mean): Negativo
Texto: Serviço de entrega péssimo, mas o produto em si é razoável. | Predição (mean): Negativo
Texto: Vale cada centavo, compraria novamente sem dúvida. | Predição (mean): Positivo
